In [1]:
from mava.networks.retention import MultiScaleRetention
from omegaconf import DictConfig
import jax
import jax.numpy as jnp
import copy

jax.config.update("jax_enable_x64", True)

bsz = 16
num_agents = 4
obs_dim = 11
num_time_steps = 400
seq_len = num_agents * num_time_steps

retnet_embed_dim = 32
retnet_num_heads = 2

2025-02-26 14:28:22.826313: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.4 which is older than the ptxas CUDA version (12.8.61). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
/home/ruanjohn/miniconda3/envs/mava/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
memory_config = DictConfig(
    {
        "type": "rec_sable",
        "decay_scaling_factor": 0.3,
        "timestep_positional_encoding": True,
        "timestep_chunk_size": None,
    }
)

decay_kappas = 1 - jnp.exp(jnp.linspace(jnp.log(1 / 32), jnp.log(1 / 512), retnet_num_heads))
decay_kappas = jnp.log(decay_kappas * memory_config.decay_scaling_factor)
decay_kappas = decay_kappas[None, :, None, None]

In [3]:
msr = MultiScaleRetention(
    embed_dim=retnet_embed_dim,
    n_head=retnet_num_heads,
    n_agents=num_agents,
    memory_config=memory_config,
    masked=False,
    decay_scaling_factor=0.3
)

In [4]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)

obs = jax.random.normal(subkey, (bsz, seq_len, retnet_embed_dim))

# assuming no resets
dones = jnp.zeros((bsz, seq_len), dtype=bool)

init_hstate = jnp.zeros((bsz, retnet_num_heads, retnet_embed_dim//retnet_num_heads, retnet_embed_dim//retnet_num_heads))
init_scale = jnp.ones((bsz, retnet_num_heads, 1, 1))
step_counts = jnp.arange(num_time_steps)
step_counts = step_counts[None, ...].repeat(bsz, axis=0)[..., None].repeat(num_agents, axis=-1)
step_counts = step_counts.reshape(bsz, seq_len)

In [5]:
key, init_key = jax.random.split(key)
params = msr.init(
    init_key,
    obs,
    obs,
    obs,
    init_hstate,
    dones,
    step_counts,
)
# step = 1
# params = msr.init(
#     init_key,
#     obs[:, step*num_agents:(step+1)*num_agents, ...],
#     obs[:, step*num_agents:(step+1)*num_agents, ...],
#     obs[:, step*num_agents:(step+1)*num_agents, ...],
#     init_hstate,
#     dones[:, step*num_agents:(step+1)*num_agents],
#     step_counts[:, step*num_agents:(step+1)*num_agents],
#     method="recurrent",
# )

In [6]:
hstate = copy.deepcopy(init_hstate)
scale = copy.deepcopy(init_scale)
act_output = []


# for the decoder we use the chunkwise
for step in range(num_time_steps):

    # todo: reset later

    updated_scale = scale * jnp.exp(decay_kappas) + 1
    scale_factor = jnp.sqrt(scale) * jnp.exp(decay_kappas) / jnp.sqrt(updated_scale)
    hstate = hstate * scale_factor
    scale = updated_scale

    obs_i = obs[:, step*num_agents:(step+1)*num_agents, ...]
    dones_i = dones[:, step*num_agents:(step+1)*num_agents]
    step_counts_i = step_counts[:, step*num_agents:(step+1)*num_agents]

    out, hstate = msr.apply(params, obs_i, obs_i, obs_i, hstate, step_counts_i, scale, method="recurrent")
    act_output.append(out)

In [7]:
act_output = jnp.concatenate(act_output, axis=1)

In [8]:
hstate = copy.deepcopy(init_hstate)
scale = copy.deepcopy(init_scale)
train_out, _ = msr.apply(params, obs, obs, obs, hstate, dones, step_counts)

In [9]:
train_out.shape

(16, 1600, 32)

In [10]:
total_error = jnp.mean(jnp.abs(train_out - act_output))
total_error

Array(1.35005074e-05, dtype=float64)

In [11]:
jnp.abs(train_out - act_output)

Array([[[2.25418745e-06, 2.19689153e-06, 8.52105770e-07, ...,
         2.51767400e-06, 2.27183121e-06, 1.62251624e-06],
        [1.99848202e-06, 1.86091776e-05, 8.02936226e-06, ...,
         3.88673996e-06, 1.05383491e-05, 8.78813235e-06],
        [9.32788015e-06, 4.12493635e-06, 1.72894605e-06, ...,
         7.93469673e-06, 5.02971938e-06, 4.49943147e-06],
        ...,
        [1.79329791e-06, 1.83861087e-05, 9.18362340e-06, ...,
         6.77952224e-06, 4.19930442e-06, 1.13301053e-05],
        [9.89306040e-06, 5.09228492e-06, 6.01536859e-06, ...,
         8.46935600e-06, 3.21245990e-05, 1.69394051e-05],
        [4.06938767e-06, 8.58491052e-06, 4.17456764e-06, ...,
         3.87192894e-06, 3.02978502e-07, 7.36397702e-07]],

       [[7.80749928e-06, 1.37802690e-06, 9.25194783e-06, ...,
         5.11021879e-06, 5.65910542e-06, 2.41408497e-06],
        [6.25627219e-06, 1.03966450e-05, 7.53551735e-06, ...,
         9.35052150e-06, 1.20680618e-05, 2.82248893e-06],
        [1.05879996e-05, 